In [9]:
from langchain.schema import BaseRetriever

In [10]:
# rag_system.py
"""
LangChain v0.3 RAG System (class-oriented) using ChromaDB.

Features:
 - Loads hierarchical JSON files under json_input_root (expects structure: json_data/dbe_<TICKER>/*.json).
 - Ingests into parent documents + child chunks (paragraphs, tables).
 - Multi-representation indexing using MultiVectorRetriever (child chunk vectors -> parent doc store).
 - Additional retrievers: dense (chroma), MultiQueryRetriever, MergerRetriever, HyDE, Decomposition, Step-Back.
 - Prompt versioning (PromptManager) and experiment logging (ExperimentLogger).
 - Evaluation helpers (EM / F1).
"""

import os
import json
import uuid
import time
import logging
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime
from config_loader import *

# LangChain v0.3 imports (see docs)
from langchain.schema import BaseRetriever
from langchain.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.schema.document  import Document as LCDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.merger_retriever import MergerRetriever
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryByteStore

# small local sparse retriever for complementing dense retrieval
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [11]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")



In [ ]:
class RAGConfig:
    """Environment Config"""
    JSON_INPUT_ROOT: Path = Path("./json_data")         
    CHROMA_PERSIST_DIR: Path = Path("./chroma_db_small")   
    CHROMA_COLLECTION_NAME: str = "rag_child_chunks"
    SUMMARIES_COLLECTION: str = "rag_summaries"        
    PROMPTS_DIR: Path = Path("./prompts_versions")
    EXPERIMENTS_DIR: Path = Path("./experiments")
    MODEL_NAME: str = "gpt-4o-mini"                      
    EMBEDDING_MODEL: str = "text-embedding-3-small"
    TEMPERATURE: float = 0.0
    K: int = 5                                          
    CHUNK_SIZE: int = 100                             
    CHUNK_OVERLAP: int = 200

    def __post_init__(self):
        self.PROMPTS_DIR.mkdir(parents=True, exist_ok=True)
        self.EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)
        self.CHROMA_PERSIST_DIR.mkdir(parents=True, exist_ok=True)



In [13]:
class PromptManager:
    def __init__(self, prompts_dir: Path):
        self.prompts_dir = prompts_dir
        self.prompts_dir.mkdir(parents=True, exist_ok=True)

    def register_prompt(self, name: str, template: str, description: str = "") -> str:
        pid = f"{name.replace(' ', '_')}_{int(time.time())}"
        payload = {
            "id": pid,
            "name": name,
            "template": template,
            "description": description,
            "created_at": datetime.now().isoformat()
        }
        path = self.prompts_dir / f"{pid}.json"
        path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
        logging.info(f"Prompt registered: {pid}")
        return pid

    def load_prompt(self, prompt_id: str) -> Dict[str, Any]:
        p = self.prompts_dir / f"{prompt_id}.json"
        if not p.exists():
            raise FileNotFoundError(f"Prompt id {prompt_id} not found at {p}")
        return json.loads(p.read_text(encoding="utf-8"))

    def list_prompts(self) -> List[str]:
        return [p.stem for p in sorted(self.prompts_dir.glob("*.json"))]



In [14]:
class ExperimentLogger:
    def __init__(self, experiments_dir: Path):
        self.dir = experiments_dir
        self.dir.mkdir(parents=True, exist_ok=True)
        self.index = self.dir / "index.jsonl"

    def log(self, entry: Dict[str, Any]) -> str:
        run_id = f"run_{int(time.time()*1000)}_{uuid.uuid4().hex[:6]}"
        payload = {"run_id": run_id, "timestamp": datetime.now().isoformat(), **entry}
        run_path = self.dir / f"{run_id}.json"
        run_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
        with self.index.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(payload, ensure_ascii=False) + "\n")
        logging.info(f"Experiment logged: {run_id}")
        return run_id



In [15]:
class TfidfRetriever:
  
    def __init__(self, docs: List[LCDocument], k: int = 5):
        self.docs = docs
        self.k = k
        self.texts = [d.page_content for d in docs]
        self.vectorizer = TfidfVectorizer(stop_words="english", max_features=20000)
        if len(self.texts) == 0:
            self.mat = None
        else:
            self.mat = self.vectorizer.fit_transform(self.texts)

    def get_relevant_documents(self, query: str) -> List[LCDocument]:
        if self.mat is None:
            return []
        v = self.vectorizer.transform([query])
        scores = (self.mat @ v.T).toarray().ravel()
        idx = np.argsort(scores)[::-1][: self.k]
        # return top-k (even if zero scores) but preserve docs length
        return [self.docs[i] for i in idx]


In [16]:
from typing import List

class TfidfRetrieverWrapper(BaseRetriever):
    """Wraps your existing TfidfRetriever so it works inside LangChain retrievers."""

    def __init__(self, tfidf_retriever):
        super().__init__()
        self._tfidf = tfidf_retriever   # your existing class instance

    def _get_relevant_documents(self, query: str) -> List[LCDocument]:
        return self._tfidf.get_relevant_documents(query)

    async def _aget_relevant_documents(self, query: str) -> List[LCDocument]:
        return self._get_relevant_documents(query)

In [ ]:
class RAGSystem:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
       
        self.llm = ChatOpenAI(model=self.cfg.MODEL_NAME, temperature=self.cfg.TEMPERATURE)
        self.embeddings = OpenAIEmbeddings(model=self.cfg.EMBEDDING_MODEL)
        self.prompt_manager = PromptManager(self.cfg.PROMPTS_DIR)
        self.experiment_logger = ExperimentLogger(self.cfg.EXPERIMENTS_DIR)

        
        self.parent_docs: List[LCDocument] = []
        self.child_docs: List[LCDocument] = []
        self.doc_id_map: Dict[str, LCDocument] = {}

        self.child_vectorstore: Optional[Chroma] = None
        self.summaries_vectorstore: Optional[Chroma] = None
        self.multi_vector_retriever: Optional[MultiVectorRetriever] = None
        self.dense_retriever = None
        self.multiquery_retriever = None
        self.merger_retriever = None
        self.tfidf_retriever = None

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.cfg.CHUNK_SIZE,
            chunk_overlap=self.cfg.CHUNK_OVERLAP
        )

    def ingest_documents(self, json_root: Path):
        
        json_root = Path(json_root)
        if not json_root.exists():
            raise FileNotFoundError(f"{json_root} does not exist")

        self.parent_docs = []
        self.child_docs = []
        id_key = "doc_id"

        for dbe_dir in sorted(json_root.glob("dbe_*")):
            if not dbe_dir.is_dir():
                continue
            ticker = dbe_dir.name.replace("dbe_", "", 1)
            for jfile in sorted(dbe_dir.glob("*.json")):
                data = json.loads(jfile.read_text(encoding="utf-8"))
                file_label = data.get("file", jfile.name)
                
                parts = []
                sections = data.get("sections") or data.get("hierarchy") or []
                for sec in sections:
                    sec_title = sec.get("title") or sec.get("section") or ""
                    parts.append(f"SECTION: {sec_title}")
                    
                    for p in sec.get("paragraphs", []):
                        parts.append(p)
                    # subsections
                    for sub in sec.get("subsections", []):
                        sub_title = sub.get("title") or sub.get("subsection") or ""
                        parts.append(f"SUBSECTION: {sub_title}")
                        for p in sub.get("paragraphs", []):
                            parts.append(p)
                        
                        for tbl in sub.get("tables", []):
                            parts.append(self._flatten_table_text(tbl))
                
                    for tbl in sec.get("tables", []):
                        parts.append(self._flatten_table_text(tbl))

                full_text = "\n\n".join(parts).strip() or ""
            
                doc_id = str(uuid.uuid4())
                parent_md = {
                    "ticker": ticker,
                    "file": file_label,
                    "source_path": str(jfile),
                    "doc_id": doc_id
                }
                parent_doc = LCDocument(page_content=full_text, metadata=parent_md)
                self.parent_docs.append(parent_doc)
                self.doc_id_map[doc_id] = parent_doc

                idx_counter = 0
                for sec in sections:
                    sec_title = sec.get("title") or sec.get("section") or ""
                    # paragraphs at section
                    for p in sec.get("paragraphs", []):
                        child_md = dict(parent_md)
                        child_md.update({
                            id_key: doc_id,
                            "section_title": sec_title,
                            "subsection_title": None,
                            "is_table": False,
                            "child_index": idx_counter
                        })
                        idx_counter += 1
                        self.child_docs.append(LCDocument(page_content=p, metadata=child_md))
                    # subsections
                    for sub in sec.get("subsections", []):
                        sub_title = sub.get("title") or sub.get("subsection") or ""
                        for p in sub.get("paragraphs", []):
                            child_md = dict(parent_md)
                            child_md.update({
                                id_key: doc_id,
                                "section_title": sec_title,
                                "subsection_title": sub_title,
                                "is_table": False,
                                "child_index": idx_counter
                            })
                            idx_counter += 1
                            self.child_docs.append(LCDocument(page_content=p, metadata=child_md))
                        for t in sub.get("tables", []):
                            tbl_text = self._flatten_table_text(t)
                            child_md = dict(parent_md)
                            child_md.update({
                                id_key: doc_id,
                                "section_title": sec_title,
                                "subsection_title": sub_title,
                                "is_table": True,
                                "table_title": t.get("title"),
                                "child_index": idx_counter
                            })
                            idx_counter += 1
                            self.child_docs.append(LCDocument(page_content=tbl_text, metadata=child_md))
                    # tables at section level
                    for t in sec.get("tables", []):
                        tbl_text = self._flatten_table_text(t)
                        child_md = dict(parent_md)
                        child_md.update({
                            id_key: doc_id,
                            "section_title": sec_title,
                            "subsection_title": None,
                            "is_table": True,
                            "table_title": t.get("title"),
                            "child_index": idx_counter
                        })
                        idx_counter += 1
                        self.child_docs.append(LCDocument(page_content=tbl_text, metadata=child_md))

        logging.info(f"Ingested parent_docs={len(self.parent_docs)}, child_docs={len(self.child_docs)}")
        #use splitter
        self.parent_docs = self.splitter.split_documents(self.parent_docs)
        self.child_docs = self.splitter.split_documents(self.child_docs)

        logging.info(f"Ingested splitted parent_docs={len(self.parent_docs)}, child_docs={len(self.child_docs)}")


    def _flatten_table_text(self, table: Dict[str, Any]) -> str:
        
        lines = []
        title = table.get("title") or ""
        if title:
            lines.append(f"TABLE: {title}")

        columns = table.get("columns") or table.get("headers") or []
        if columns:
            
            col_labels = [c.get("label") if isinstance(c, dict) else str(c) for c in columns]
            lines.append("COLUMNS: " + " | ".join(col_labels))

        rows = table.get("rows") or table.get("data") or []
        for r in rows:
            if isinstance(r, dict):
                label = r.get("label") or r.get("row_header") or ""
                
                cells = r.get("cells") or r.get("values") or {}
                if isinstance(cells, dict):
                    kvs = [f"{k}: {v}" for k, v in cells.items()]
                    lines.append(f"{label} -> " + "; ".join(kvs))
                elif isinstance(cells, list):
                    lines.append(f"{label} -> " + " | ".join([str(x) for x in cells]))
                else:
                    
                    lines.append(json.dumps(r, ensure_ascii=False))
            elif isinstance(r, list):
               
                lines.append(" | ".join([str(x) for x in r]))
            else:
                lines.append(str(r))
        return "\n".join(lines)


    def build_indexes(self, persist: bool = False):
        """Builds the Chroma vectorstore for child docs and a MultiVectorRetriever mapping child vectors -> parent docs."""
        # sanity
        if not self.child_docs or not self.parent_docs:
            raise RuntimeError("No documents ingested. Run ingest_documents() first.")
        
        if persist:
            print("persist: ",persist)
            logging.info(f"Child embedding is building up....{len(self.child_docs)}")
           
            self.child_vectorstore = Chroma.from_documents(
            documents=self.child_docs,
            collection_name=self.cfg.CHROMA_COLLECTION_NAME,
            embedding=self.embeddings,
            persist_directory=str(self.cfg.CHROMA_PERSIST_DIR)
        )
        else:
            logging.info("Child embedding picked up from child vectorstore")
            self.child_vectorstore = Chroma(
            persist_directory=str(self.cfg.CHROMA_PERSIST_DIR),
            collection_name=self.cfg.CHROMA_COLLECTION_NAME,
            embedding_function=self.embeddings
        )
        child_vs=self.child_vectorstore
        logging.info("Child vectorstore built and persisted.")
      
        self.dense_retriever = child_vs.as_retriever(search_kwargs={"k": self.cfg.K})
        logging.info("1) Dense retriever (simple as_retriever)")
       
        self.tfidf_retriever = TfidfRetriever(self.child_docs, k=self.cfg.K)
        self.wrapped_tfidf = TfidfRetrieverWrapper(self.tfidf_retriever)
        logging.info("2) TF-IDF retriever (sparse)")

        try:
            self.multiquery_retriever = MultiQueryRetriever.from_llm(
                retriever=self.dense_retriever,
                llm=self.llm
                )
            logging.info(" 3) MultiQuery retriever (uses LLM to expand queries) done.")
        except Exception:
            
            logging.info("fallback: use dense retriever.")
            self.multiquery_retriever = self.dense_retriever

        self.merger_retriever = MergerRetriever(retrievers=[self.dense_retriever, self.wrapped_tfidf])
        logging.info("4) Merger retriever (merges dense + sparse)")
        """
        summaries = []
        doc_ids = []
        for parent in self.parent_docs:
            doc_id = parent.metadata.get("doc_id")
            doc_ids.append(doc_id)
            
            summarizer_template = (
                "Summarize the following filing in 2-3 short sentences focusing on key financial tables and amounts:\n\n"
                "{doc}"
            )
            prompt = ChatPromptTemplate.from_template(summarizer_template)
            chain = prompt | self.llm
            try:
                summary = chain.invoke({"doc": parent.page_content})
            except Exception:
                
                summary = parent.page_content[:1000]
            
            summary_text = str(summary).strip()
            summaries.append(LCDocument(page_content=summary_text, metadata={"doc_id": doc_id}))
   
        if persist:
            logging.info("Summary embedding is building up....")
            self.summaries_vectorstore = Chroma.from_documents(
            documents=summaries,
            collection_name=self.cfg.SUMMARIES_COLLECTION,
            embedding=self.embeddings,
            persist_directory=str(self.cfg.CHROMA_PERSIST_DIR)
        )
        else:
            logging.info("Summary embedding is picked up from vector stores..")
            self.summaries_vectorstore = Chroma(
            persist_directory=str(self.cfg.CHROMA_PERSIST_DIR),
            collection_name=self.cfg.SUMMARIES_COLLECTION,
            embedding_function=self.embeddings
        )
        summaries_vs=self.summaries_vectorstore

       
        byte_store = InMemoryByteStore()

        parent_pairs = [(p.metadata.get("doc_id"), p) for p in self.parent_docs]
        byte_store.mset(parent_pairs)

        mv_retriever = MultiVectorRetriever(
            vectorstore=summaries_vs,
            byte_store=byte_store,
            id_key="doc_id",
            search_kwargs={"k": self.cfg.K}
        )
        # note: we could add other vectors (hypothetical questions) to the same summaries_vs collection if desired
        self.multi_vector_retriever = mv_retriever
        """

        logging.info("Indexes and retrievers built: dense, multi-query, merger, multi-vector, tfidf (sparse).")

    
    def _hyde_retrieve(self, question: str, k: Optional[int] = None) -> List[LCDocument]:
        """HyDE: generate a hypothetical answer paragraph, embed it, then similarity search on child vectorstore."""
        k = k or self.cfg.K
        hyde_prompt = ChatPromptTemplate.from_template(
            "Write a short factual paragraph that directly answers the question (no sources). Question: {question}"
        )
        chain = hyde_prompt | self.llm
        hyde_text = chain.invoke({"question": question})
        hyde_text = str(hyde_text)
      
        vec = self.embeddings.embed_query(hyde_text)
        docs = self.child_vectorstore.similarity_search_by_vector(vec, k=k)
        return docs

    def _decomposition_retrieve(self, question: str, k_each: int = 4) -> List[LCDocument]:
        """Ask the LLM to decompose the question into sub-questions, retrieve for each, union results."""
        subq_prompt = ChatPromptTemplate.from_template(
            "Decompose the question into 3-5 focused sub-questions (one per line):\n\nQuestion: {question}"
        )
        chain = subq_prompt | self.llm
        subq_text = chain.invoke({"question": question})
        subq_text = str(subq_text)
        subqs = [s.strip("-. \t") for s in subq_text.splitlines() if s.strip()]
        all_docs = []
        seen = set()
        for sq in subqs:
            docs = self.dense_retriever.get_relevant_documents(sq) if hasattr(self.dense_retriever, "get_relevant_documents") else self.dense_retriever(sq)
            for d in docs:
                key = (d.metadata.get("source_path"), d.page_content[:200])
                if key not in seen:
                    all_docs.append(d)
                    seen.add(key)
        return all_docs

    # ---------- Answer synthesis ----------
    def _synthesize_answer(self, docs: List[LCDocument], question: str, prompt_template: Optional[str] = None) -> Tuple[str, List[Dict[str, Any]]]:
        """Produce final answer with LLM and return (answer_text, provenance)."""
        # default prompt
        if prompt_template is None:
            prompt_template = (
                "You are an expert financial assistant. Use the context below to answer the question concisely and factually.\n\n"
                "Context:\n{context}\n\nQuestion: {question}\n\nAnswer (short, cite sources inline as [Source X]):"
            )
        # build context string with provenance headers up to top-K
        context_lines = []
        for i, d in enumerate(docs[: self.cfg.K ]):
            md = d.metadata or {}
            src = md.get("source_path") or md.get("file") or md.get("ticker") or "unknown"
            header = f"[Source {i+1}] {src} (ticker={md.get('ticker')}, section={md.get('section_title')}, subsection={md.get('subsection_title')})"
            context_lines.append(header + "\n" + d.page_content)
        context_block = "\n\n---\n\n".join(context_lines) or "No context found."

        prompt = ChatPromptTemplate.from_template(prompt_template)
        chain = prompt | self.llm
        try:
            out = chain.invoke({"context": context_block, "question": question})
            answer = str(out).strip()
        except Exception as ex:
            # fallback: make a simple call
            answer = f"(LLM error): {ex}"

        provenance = [{"source": (d.metadata.get("source_path") or d.metadata.get("file")), "metadata": d.metadata} for d in docs]
        return answer, provenance

    # ---------- Routing & Orchestration ----------
    def route_strategy(self, question: str, options: List[str]) -> str:
        """Ask LLM to pick a strategy name from options (deterministic intent, temp=0)."""
        prompt = (
            "You are a routing assistant. Choose exactly one best retrieval strategy (return only the exact name):\n\n"
            f"Options: {', '.join(options)}\n\nQuestion: {question}\n\nReturn one option name exactly."
        )
        chain = ChatPromptTemplate.from_template("{q}") | self.llm
        choice = chain.invoke({"q": prompt})
        choice = str(choice).strip()
        # validate
        if choice not in options:
            # attempt matching substring
            for o in options:
                if o.lower() in choice.lower():
                    return o
            return options[0]
        return choice

    def run_query(self, question: str, strategy: Optional[str] = None, prompt_id: Optional[str] = None, extra: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        """
        High-level entry to run a single question.
        strategy options: ['naive','multi_query','fusion','hyde','decompose','step_back','raptor','multi_vector','auto']
        If prompt_id is provided, the prompt template from PromptManager is used for synthesis.
        """
        extra = extra or {}
        strategies = ['naive', 'multi_query', 'fusion', 'hyde', 'decompose', 'step_back', 'raptor', 'multi_vector']
        if strategy is None or strategy == "auto":
            chosen = self.route_strategy(question, strategies)
            logging.info(f"Router chose strategy: {chosen}")
            strategy = chosen

        prompt_template = None
        prompt_version = "default"
        if prompt_id:
            info = self.prompt_manager.load_prompt(prompt_id)
            prompt_template = info.get("template")
            prompt_version = prompt_id

        # retrieval stage
        docs = []
        if strategy == "naive":
            docs = self.dense_retriever.get_relevant_documents(question)
        elif strategy == "multi_query":
            docs = self.multiquery_retriever.get_relevant_documents(question)
        elif strategy == "fusion":
            # MergerRetriever: merges dense + tfidf
            docs = self.merger_retriever.get_relevant_documents(question)
        elif strategy == "hyde":
            docs = self._hyde_retrieve(question)
        elif strategy == "decompose":
            docs = self._decomposition_retrieve(question)
        elif strategy == "step_back":
            # initial answer with naive, then follow-ups
            initial_docs = self.dense_retriever.get_relevant_documents(question)
            answer0, _ = self._synthesize_answer(initial_docs, question, prompt_template)
            followup_prompt = ChatPromptTemplate.from_template(
                "You produced the following answer:\n\n{ans}\n\nGenerate up to 3 specific follow-up queries that would help obtain missing evidence (one per line)."
            )
            chain = followup_prompt | self.llm
            followup_text = chain.invoke({"ans": answer0})
            followups = [l.strip("-. \t") for l in str(followup_text).splitlines() if l.strip()]
            extra_docs = []
            for fq in followups:
                extra_docs.extend(self.dense_retriever.get_relevant_documents(fq))
            # union & dedupe top
            seen = set(); merged=[]
            for d in (initial_docs + extra_docs):
                key = (d.metadata.get("source_path"), d.page_content[:200])
                if key not in seen:
                    merged.append(d); seen.add(key)
            docs = merged
        elif strategy == "raptor":
            top_parents = self.multi_vector_retriever.invoke(question)  # returns parent Documents
            
            parent_ids = [p.metadata.get("doc_id") for p in top_parents][:3]
           
            candidate_children = [c for c in self.child_docs if c.metadata.get("doc_id") in parent_ids]
            
            q_vec = self.embeddings.embed_query(question)
            
            raw_hits = self.child_vectorstore.similarity_search(question, k=self.cfg.K*3)
            filtered = [d for d in raw_hits if d.metadata.get("doc_id") in parent_ids]
            docs = filtered[: self.cfg.K]
        elif strategy == "multi_vector":
            
            parent_hits = self.multi_vector_retriever.invoke(question)
            
            docs = []
            for p in parent_hits[: self.cfg.K]:
                pid = p.metadata.get("doc_id")
                # get top child chunks for the parent (use child_vectorstore with filter)
                child_hits = self.child_vectorstore.similarity_search(question, k=4)
                # filter
                child_for_parent = [c for c in child_hits if c.metadata.get("doc_id") == pid]
                docs.extend(child_for_parent)
            if not docs:
                # fallback to dense retriever global
                docs = self.dense_retriever.get_relevant_documents(question)
        else:
            # default naive
            docs = self.dense_retriever.get_relevant_documents(question)

        # answer generation
        answer_text, provenance = self._synthesize_answer(docs, question, prompt_template)

        # log experiment
        log_entry = {
            "question": question,
            "strategy": strategy,
            "prompt_version": prompt_version,
            "model_name": self.cfg.MODEL_NAME,
            "k_retrieved": len(docs),
            "retrieved_docs": [{"source": d.metadata.get("source_path") or d.metadata.get("file"), "metadata": d.metadata} for d in docs],
            "answer": answer_text
        }
        run_id = self.experiment_logger.log(log_entry)
        return {"run_id": run_id, "answer": answer_text, "provenance": provenance, "retrieved_count": len(docs)}

    # ---------- Evaluation helpers ----------
    @staticmethod
    def normalize_answer(s: str) -> str:
        s = s.lower().strip()
        import re
        s = re.sub(r"[^a-z0-9]+", " ", s)
        return " ".join(s.split())

    @staticmethod
    def exact_match(s1: str, s2: str) -> bool:
        return RAGSystem.normalize_answer(s1) == RAGSystem.normalize_answer(s2)

    @staticmethod
    def f1_score(pred: str, gold: str) -> float:
        p_tokens = RAGSystem.normalize_answer(pred).split()
        g_tokens = RAGSystem.normalize_answer(gold).split()
        if not p_tokens or not g_tokens:
            return 0.0
        common = set(p_tokens) & set(g_tokens)
        if not common:
            return 0.0
        prec = len(common) / len(p_tokens)
        rec = len(common) / len(g_tokens)
        if prec + rec == 0:
            return 0.0
        return 2 * prec * rec / (prec + rec)

    def evaluate_dataset(self, qa_pairs: List[Dict[str, str]], strategy: str = "auto", prompt_id: Optional[str] = None) -> Dict[str, Any]:
        results = []
        for q in qa_pairs:
            out = self.run_query(q["question"], strategy=strategy, prompt_id=prompt_id)
            pred = out["answer"]
            em = 1 if self.exact_match(pred, q["answer"]) else 0
            f1 = self.f1_score(pred, q["answer"])
            results.append({"question": q["question"], "gold": q["answer"], "pred": pred, "em": em, "f1": f1, "run_id": out["run_id"]})
        avg_em = sum(r["em"] for r in results) / len(results)
        avg_f1 = sum(r["f1"] for r in results) / len(results)
        return {"results": results, "avg_em": avg_em, "avg_f1": avg_f1}




In [18]:
cfg = RAGConfig()
rag = RAGSystem(cfg)
rag.ingest_documents(cfg.JSON_INPUT_ROOT)

C:\Users\Omen\AppData\Local\Temp\ipykernel_1588\1207220022.py:5: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  self.llm = ChatOpenAI(model=self.cfg.MODEL_NAME, temperature=self.cfg.TEMPERATURE)
2025-08-21 10:45:28,891 INFO Ingested parent_docs=1, child_docs=2441
2025-08-21 10:45:29,150 INFO Ingested splitted parent_docs=1663, child_docs=3424


In [19]:
rag.build_indexes(persist=True)
#rag.build_indexes()

2025-08-21 10:45:29,173 INFO Child embedding is building up....3424
2025-08-21 10:45:29,338 INFO Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


persist:  True


2025-08-21 10:45:35,078 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-21 10:45:41,792 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-21 10:45:49,987 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-21 10:45:54,849 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-21 10:46:03,917 INFO Child vectorstore built and persisted.
2025-08-21 10:46:03,917 INFO 1) Dense retriever (simple as_retriever)
2025-08-21 10:46:04,141 INFO 2) TF-IDF retriever (sparse)
2025-08-21 10:46:04,147 INFO  3) MultiQuery retriever (uses LLM to expand queries) done.
2025-08-21 10:46:04,147 INFO 4) Merger retriever (merges dense + sparse)
2025-08-21 10:46:04,147 INFO Indexes and retrievers built: dense, multi-query, merger, multi-vector, tfidf (sparse).


In [20]:
rag.dense_retriever.vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001FF1037FF50>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001FF10430080>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization='org-8XwQuUkcdwOPFzPXvmJFEDhE', allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [31]:
rag.child_docs[0].metadata

{'ticker': 'MMM',
 'file': '10-K_2025-02-05_000006674025000006_primary.htm',
 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json',
 'doc_id': '24c4d304-e0e4-4c92-a955-8672c49bc48d',
 'section_title': 'Document',
 'subsection_title': None,
 'is_table': False,
 'child_index': 0}

In [41]:
indx=368
child_docs = [d for d in rag.child_docs if d.metadata.get("child_index") == indx]

combined_text = "\n".join(doc.page_content for doc in child_docs)

print(combined_text)  # print only first 1000 chars for preview


worldwide economic, political, regulatory, international trade, geopolitical, capital markets and other external conditions and other factors beyond the Company's control, including inflation; recession; military conflicts; trade restrictions such as sanctions, tariffs, and retaliatory measures; regulatory requirements, legal actions, or enforcement; and natural and other disasters or climate change affecting the operations of the Company or its customers and suppliers,


In [21]:
a=[i.metadata['source_path'] for i in rag.child_docs if i.metadata['source_path']!='json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json']

In [22]:
set(a)

set()

In [27]:
# register a prompt version
###    "You are an expert financial assistant. Use the context to answer the question.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer (cite sources as [Source X]):"
#)

default_prompt ="""
You are an internal Adobe financial and business intelligence analyst.
Your task is to carefully read the following document excerpt and extract only the 
information that is relevant to the question. 

The excerpt may contain:
- Textual explanations (e.g., market commentary, strategic plans)
- Data tables (e.g., revenue breakdown, customer growth metrics)
- Charts (converted to text in this format)

Guidelines:
- Focus on metrics, financial figures, percentages, growth rates, and trends.
- Keep “Challenge Context” and “Business Value Context” under 20 words each.
- Infer ticker and company name from the context (including any visible headers/footers/metadata). If still unclear, leave them blank.
- If a table is described, extract key numbers with their labels.
- If a chart is described, summarize the main trend or pattern.
- If no relevant information is found, return "No relevant information".
- Avoid speculation or outside knowledge — use ONLY the excerpt.
- Highlight numbers, dates, trends, and causes when present.
- Identify filing type (10-K, 10-Q, 8-K, DEF 14A, etc.) and filing date if present in the snippet text. If absent, leave blank.
- Keep sentences concise but informative.

---

Question:
{question}

Relevant Findings from this excerpt:
"""

pid = rag.prompt_manager.register_prompt("fin_v1", default_prompt, "Default financial response prompt")

# run an example query
#q = "what is the ticker, company name,five risk factors"
q=" Please identify the following Ticker , Company Name ,headcount growth, Challenge Summary , Challenge Context , Business Value , Business Value Context , Department , Adobe Solution , Filing Type , Filing Date , Source Quote , Source Section ,"
strategies = ['naive', 'multi_query', 'fusion', 'hyde', 'decompose', 'step_back']
final_output=[]
out={}
for strtegy in strategies:
    print("="*30)
    print("strtegy:", strtegy)
    print("\n")
    
    out = rag.run_query(q, strategy=strtegy, prompt_id=pid)
    if "No relevant information." not in out["answer"]:
        final_output.append(out["answer"])
        out[strtegy]=out["answer"]

    print("Run ID:", out["run_id"])
    print("Answer:\n", out["answer"])
    print("Provenance (first 3):", out["provenance"][:3])
    print("="*30)
    print("\n"*2)


print("FINAL ANSWER:\n", final_output)


2025-08-21 11:06:44,407 INFO Prompt registered: fin_v1_1755754604


strtegy: naive




2025-08-21 11:06:45,442 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-21 11:06:46,359 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-21 11:06:46,479 INFO Experiment logged: run_1755754606359_207d3d


Run ID: run_1755754606359_207d3d
Answer:
 content='No relevant information.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 317, 'total_tokens': 321, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_560af6e559', 'finish_reason': 'stop', 'logprobs': None} id='run--604a259f-f944-473f-8642-7ad0703d5f54-0'
Provenance (first 3): [{'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'section_title': 'Document', 'child_index': 2438, 'is_table': True, 'ticker': 'MMM', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'doc_id': '24c4d304-e0e4-4c92-a955-8672c49bc48d', 'file': '10-K_2025-02-05_000006674025000006_primary.htm'}}, {'source': 'json_data\\dbe_MMM\\10-

2025-08-21 11:06:50,432 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-21 11:06:50,461 INFO Generated queries: ['1. Can you provide details on the ticker symbol, company name, headcount growth, and a summary of the challenges faced, including their context and business value, along with the relevant department, Adobe solution, filing type, filing date, and source information?', '2. I need information regarding the ticker, company name, growth in headcount, a summary of challenges and their context, business value and context, the department involved, the Adobe solution used, as well as the filing type, date, and source quotes or sections.', '3. Please gather the following information: ticker symbol, company name, headcount growth, a summary of challenges and their context, business value and its context, the relevant department, the Adobe solution implemented, along with the filing type, filing date, and any quotes or sections from the sour

...

Run ID: run_1755754619988_00448c
Answer:
 content='No relevant information.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 317, 'total_tokens': 321, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None} id='run--192caf7f-610a-4384-8c49-5347e24b5053-0'
Provenance (first 3): [{'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'subsection_title': 'Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations', 'doc_id': '24c4d304-e0e4-4c92-a955-8672c49bc48d', 'child_index': 936, 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'is_table': True, 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_0

2025-08-21 11:07:02,146 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-21 11:07:02,592 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-21 11:07:03,276 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-21 11:07:03,304 INFO Experiment logged: run_1755754623276_26bb74


Run ID: run_1755754623276_26bb74
Answer:
 content='No relevant information.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 317, 'total_tokens': 321, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None} id='run--16773eea-a1ec-475c-afa2-e39abe97c502-0'
Provenance (first 3): [{'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'ticker': 'MMM', 'section_title': 'Document', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'is_table': True, 'subsection_title': 'Item 8. Financial Statements and Supplementary Data', 'child_index': 2145, 'doc_id': '2

2025-08-21 11:07:03,842 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-21 11:07:04,601 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-21 11:07:06,642 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-21 11:07:07,359 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-21 11:07:07,985 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-21 11:07:07,992 INFO Experiment logged: run_1755754627985_ab8499


Run ID: run_1755754627985_ab8499
Answer:
 content='No relevant information.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 317, 'total_tokens': 321, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_560af6e559', 'finish_reason': 'stop', 'logprobs': None} id='run--da54f8ba-4ee4-466f-8089-484740b8764c-0'
Provenance (first 3): [{'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'ticker': 'MMM', 'doc_id': '24c4d304-e0e4-4c92-a955-8672c49bc48d', 'child_index': 2438, 'section_title': 'Document', 'is_table': True}}, {'source': 'json_data\\dbe_MMM\\10-

In [30]:
rag.child_docs[1]

Document(metadata={'ticker': 'MMM', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'doc_id': '24c4d304-e0e4-4c92-a955-8672c49bc48d', 'section_title': 'Document', 'subsection_title': None, 'is_table': False, 'child_index': 0}, page_content='mmm:defendant utr:mi mmm:chemical mmm:individual mmm:age mmm:segment mmm:division 0000066740 2024-01-01 2024-12-31 0000066740 exch:XNYS us-gaap:CommonStockMember 2024-01-01 2024-12-31 0000066740 exch:XCHI us-gaap:CommonStockMember 2024-01-01 2024-12-31 0000066740 exch:XNYS mmm:Notes1500PercentDue2026Member 2024-01-01 2024-12-31 0000066740 exch:XNYS mmm:Notes1750PercentDue2030Member 2024-01-01 2024-12-31 0000066740 exch:XNYS mmm:Notes1.500PercentDue2031Member 2024-01-01 2024-12-31 0000066740 2024-06-30 0000066740 2025-01-31 0000066740 2023-01-01 2023-12-31 0000066740 2022-01-01 2022-12-31 0000066740 2024-12-31 0000066740 2023-12-31 0000066740 2021-12-31 00

In [28]:
d=rag.child_vectorstore.get('24c4d304-e0e4-4c92-a955-8672c49bc48d')
d

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [24]:
2025-08-20 12:21:46,278 INFO Prompt registered: fin_v1_1755672706
==============================
strtegy: naive


2025-08-20 12:21:46,928 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-20 12:21:49,458 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:21:49,474 INFO Experiment logged: run_1755672709474_fcff5a
Run ID: run_1755672709474_fcff5a
Answer:
 content='- **Ticker**: MMM\n- **Company Name**: 3M Company\n- **Headcount Growth**: No relevant information\n- **Challenge Summary**: No relevant information\n- **Challenge Context**: No relevant information\n- **Business Value**: No relevant information\n- **Business Value Context**: No relevant information\n- **Department**: No relevant information\n- **Adobe Solution**: No relevant information\n- **Filing Type**: 10-K\n- **Filing Date**: 2025-02-05\n- **Source Quote**: No relevant information\n- **Source Section**: Item 8. Financial Statements and Supplementary Data' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 141, 'prompt_tokens': 2632, 'total_tokens': 2773, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None} id='run--cb87dc4b-98e2-47ef-8ab1-832a95cdb4a6-0'
Provenance (first 3): [{'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'is_table': True, 'section_title': 'Document', 'child_index': 2199, 'ticker': 'MMM', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'subsection_title': 'Item 8. Financial Statements and Supplementary Data'}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'ticker': 'MMM', 'child_index': 2138, 'section_title': 'Document', 'is_table': True, 'doc_id': '0d207374-f0d3-4b51-babd-9ba9678f15cf', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'subsection_title': 'Item 8. Financial Statements and Supplementary Data'}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'is_table': True, 'ticker': 'MMM', 'subsection_title': 'Item 16. Form 10-K Summary', 'child_index': 2433, 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'section_title': 'Document'}}]
==============================



==============================
strtegy: multi_query


2025-08-20 12:21:52,041 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:21:52,041 INFO Generated queries: ['1. Can you provide details on the ticker symbol, company name, headcount growth, and a summary of the challenges faced, including their context and business value, along with the relevant department, Adobe solution, filing type, filing date, and source information?', '2. I need information regarding the ticker, company name, growth in headcount, a summary of challenges and their context, business value and context, the department involved, the Adobe solution used, as well as the filing type, date, and source quotes or sections.', '3. Please gather the following information: ticker symbol, company name, headcount growth, a summary of challenges and their context, business value and its context, the relevant department, the Adobe solution implemented, along with the filing type, filing date, and any quotes or sections from the source.']
2025-08-20 12:21:52,963 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-20 12:21:53,580 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-20 12:21:54,813 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-20 12:21:55,532 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:21:55,534 INFO Experiment logged: run_1755672715534_030054
Run ID: run_1755672715534_030054
Answer:
 content='No relevant information.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 2445, 'total_tokens': 2449, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None} id='run--58c31a19-d4c9-47b8-af06-07f55c764a6b-0'
Provenance (first 3): [{'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'subsection_title': 'Item 15. Exhibits, Financial Statement Schedules', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'ticker': 'MMM', 'is_table': False, 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'section_title': 'Document', 'doc_id': 'ea3177b8-0d54-4d92-91a2-849b59f8396c', 'child_index': 2376}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'child_index': 2376, 'doc_id': '61632580-23eb-44b7-ba87-5dd9554eeefa', 'section_title': 'Document', 'ticker': 'MMM', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'is_table': False, 'subsection_title': 'Item 15. Exhibits, Financial Statement Schedules'}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'is_table': False, 'subsection_title': 'Item 15. Exhibits, Financial Statement Schedules', 'child_index': 2376, 'section_title': 'Document', 'ticker': 'MMM', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'doc_id': '7549520c-0ad9-4710-8622-8f9cd166d24c', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json'}}]
==============================



==============================
strtegy: fusion


2025-08-20 12:21:56,054 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-20 12:21:59,217 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:21:59,242 INFO Experiment logged: run_1755672719223_5aa44f
Run ID: run_1755672719223_5aa44f
Answer:
 content='- **Ticker**: MMM\n- **Company Name**: 3M Company\n- **Headcount Growth**: No relevant information\n- **Challenge Summary**: No relevant information\n- **Challenge Context**: No relevant information\n- **Business Value**: No relevant information\n- **Business Value Context**: No relevant information\n- **Department**: No relevant information\n- **Adobe Solution**: No relevant information\n- **Filing Type**: 10-K\n- **Filing Date**: 2025-02-05\n- **Source Quote**: "Refer to the Overview section for a summary of net sales by geographic area and business segment."\n- **Source Section**: Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 162, 'prompt_tokens': 2106, 'total_tokens': 2268, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None} id='run--bdf2d313-d39a-406f-b9f6-8a043322ddf0-0'
Provenance (first 3): [{'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'is_table': True, 'section_title': 'Document', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'ticker': 'MMM', 'child_index': 2199, 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6', 'subsection_title': 'Item 8. Financial Statements and Supplementary Data'}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'ticker': 'MMM', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6', 'section_title': 'Document', 'subsection_title': 'Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations', 'is_table': False, 'child_index': 820}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'is_table': True, 'subsection_title': 'Item 8. Financial Statements and Supplementary Data', 'child_index': 2138, 'doc_id': '0d207374-f0d3-4b51-babd-9ba9678f15cf', 'ticker': 'MMM', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'section_title': 'Document'}}]
==============================



==============================
strtegy: hyde


2025-08-20 12:22:01,984 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:22:02,802 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-20 12:22:06,482 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:22:06,498 INFO Experiment logged: run_1755672726482_3130f7
Run ID: run_1755672726482_3130f7
Answer:
 content='- **Ticker**: MMM\n- **Company Name**: \n- **Headcount Growth**: 36,000 (2024) vs. 50,000 (2023) in Americas; 13,500 (2024) vs. 17,000 (2023) in Asia Pacific; 12,000 (2024) vs. 18,000 (2023) in Europe, Middle East & Africa.\n- **Challenge Summary**: Employee headcount reduction across regions.\n- **Challenge Context**: Decrease in employee numbers in key markets.\n- **Business Value**: \n- **Business Value Context**: \n- **Department**: \n- **Adobe Solution**: \n- **Filing Type**: 10-K\n- **Filing Date**: 2025-02-05\n- **Source Quote**: "Total Company headcount decreased from 85,000 (2023) to 61,500 (2024)."\n- **Source Section**: Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 225, 'prompt_tokens': 6597, 'total_tokens': 6822, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None} id='run--b5226a67-31a9-4253-bf61-630e21d8a23a-0'
Provenance (first 3): [{'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'child_index': 936, 'is_table': True, 'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'ticker': 'MMM', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'section_title': 'Document', 'subsection_title': 'Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations'}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'subsection_title': 'Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations', 'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6', 'child_index': 928, 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'section_title': 'Document', 'ticker': 'MMM', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'is_table': True}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'child_index': 928, 'subsection_title': 'Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6', 'is_table': True, 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'section_title': 'Document', 'ticker': 'MMM'}}]
==============================



==============================
strtegy: decompose


2025-08-20 12:22:08,657 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:22:09,453 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-20 12:22:12,427 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:22:12,427 INFO Experiment logged: run_1755672732427_76dbbe
Run ID: run_1755672732427_76dbbe
Answer:
 content='- **Ticker**: MMM\n- **Company Name**: \n- **Headcount Growth**: Decreased from **50,000** in 2023 to **36,000** in 2024.\n- **Challenge Summary**: Headcount reduction impacting operational capacity.\n- **Challenge Context**: Decrease in workforce affecting productivity.\n- **Business Value**: \n- **Business Value Context**: \n- **Department**: \n- **Adobe Solution**: \n- **Filing Type**: 10-K\n- **Filing Date**: 2025-02-05\n- **Source Quote**: "Headcount decreased from 50,000 to 36,000."\n- **Source Section**: Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 167, 'prompt_tokens': 2837, 'total_tokens': 3004, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None} id='run--7007df8a-b297-43cd-936c-b8e46e795bdb-0'
Provenance (first 3): [{'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'is_table': True, 'subsection_title': 'Item 8. Financial Statements and Supplementary Data', 'child_index': 2145, 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'section_title': 'Document', 'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6', 'ticker': 'MMM'}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'subsection_title': 'Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'is_table': True, 'ticker': 'MMM', 'section_title': 'Document', 'child_index': 928, 'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6'}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'subsection_title': 'Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations', 'ticker': 'MMM', 'is_table': True, 'child_index': 928, 'section_title': 'Document', 'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json'}}]
==============================



==============================
strtegy: step_back


2025-08-20 12:22:13,244 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-20 12:22:15,806 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:22:17,547 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:22:18,265 INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-08-20 12:22:21,133 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-08-20 12:22:21,138 INFO Experiment logged: run_1755672741137_310633
Run ID: run_1755672741137_310633
Answer:
 content='- **Ticker**: MMM\n- **Company Name**: 3M Company\n- **Headcount Growth**: No relevant information\n- **Challenge Summary**: Operational challenges may adversely affect financial performance.\n- **Challenge Context**: Financial results depend on successful execution of business plans.\n- **Business Value**: Improved operational efficiency and productivity.\n- **Business Value Context**: Streamlining operations to enhance long-term performance.\n- **Department**: No relevant information\n- **Adobe Solution**: No relevant information\n- **Filing Type**: 10-K\n- **Filing Date**: 2025-02-05\n- **Source Quote**: "The Company’s financial results depend on the successful execution of its business operating plans."\n- **Source Section**: Item 1A. Risk Factors' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 168, 'prompt_tokens': 2391, 'total_tokens': 2559, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1792}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None} id='run--301057b4-a664-481d-a2e7-aca4e6c005f5-0'
Provenance (first 3): [{'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'is_table': True, 'ticker': 'MMM', 'subsection_title': 'Item 8. Financial Statements and Supplementary Data', 'section_title': 'Document', 'child_index': 2199, 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json'}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'subsection_title': 'Item 8. Financial Statements and Supplementary Data', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'child_index': 2138, 'section_title': 'Document', 'is_table': True, 'ticker': 'MMM', 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'doc_id': '0d207374-f0d3-4b51-babd-9ba9678f15cf'}}, {'source': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'metadata': {'is_table': True, 'source_path': 'json_data\\dbe_MMM\\10-K_2025-02-05_000006674025000006_primary.json', 'section_title': 'Document', 'child_index': 2433, 'ticker': 'MMM', 'file': '10-K_2025-02-05_000006674025000006_primary.htm', 'subsection_title': 'Item 16. Form 10-K Summary', 'doc_id': '0351ac5c-1def-475b-8a32-8b0096fe0dd6'}}]
==============================



FINAL ANSWER:
 ["content='- **Ticker**: MMM\\n- **Company Name**: 3M Company\\n- **Headcount Growth**: No relevant information\\n- **Challenge Summary**: No relevant information\\n- **Challenge Context**: No relevant information\\n- **Business Value**: No relevant information\\n- **Business Value Context**: No relevant information\\n- **Department**: No relevant information\\n- **Adobe Solution**: No relevant information\\n- **Filing Type**: 10-K\\n- **Filing Date**: 2025-02-05\\n- **Source Quote**: No relevant information\\n- **Source Section**: Item 8. Financial Statements and Supplementary Data' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 141, 'prompt_tokens': 2632, 'total_tokens': 2773, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_51db84afab', 'finish_reason': 'stop', 'logprobs': None} id='run--cb87dc4b-98e2-47ef-8ab1-832a95cdb4a6-0'", 'content=\'- **Ticker**: MMM\\n- **Company Name**: 3M Company\\n- **Headcount Growth**: No relevant information\\n- **Challenge Summary**: No relevant information\\n- **Challenge Context**: No relevant information\\n- **Business Value**: No relevant information\\n- **Business Value Context**: No relevant information\\n- **Department**: No relevant information\\n- **Adobe Solution**: No relevant information\\n- **Filing Type**: 10-K\\n- **Filing Date**: 2025-02-05\\n- **Source Quote**: "Refer to the Overview section for a summary of net sales by geographic area and business segment."\\n- **Source Section**: Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations\' additional_kwargs={} response_metadata={\'token_usage\': {\'completion_tokens\': 162, \'prompt_tokens\': 2106, \'total_tokens\': 2268, \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \'audio_tokens\': 0, \'reasoning_tokens\': 0, \'rejected_prediction_tokens\': 0}, \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cached_tokens\': 0}}, \'model_name\': \'gpt-4o-mini\', \'system_fingerprint\': \'fp_51db84afab\', \'finish_reason\': \'stop\', \'logprobs\': None} id=\'run--bdf2d313-d39a-406f-b9f6-8a043322ddf0-0\'', 'content=\'- **Ticker**: MMM\\n- **Company Name**: \\n- **Headcount Growth**: 36,000 (2024) vs. 50,000 (2023) in Americas; 13,500 (2024) vs. 17,000 (2023) in Asia Pacific; 12,000 (2024) vs. 18,000 (2023) in Europe, Middle East & Africa.\\n- **Challenge Summary**: Employee headcount reduction across regions.\\n- **Challenge Context**: Decrease in employee numbers in key markets.\\n- **Business Value**: \\n- **Business Value Context**: \\n- **Department**: \\n- **Adobe Solution**: \\n- **Filing Type**: 10-K\\n- **Filing Date**: 2025-02-05\\n- **Source Quote**: "Total Company headcount decreased from 85,000 (2023) to 61,500 (2024)."\\n- **Source Section**: Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations\' additional_kwargs={} response_metadata={\'token_usage\': {\'completion_tokens\': 225, \'prompt_tokens\': 6597, \'total_tokens\': 6822, \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \'audio_tokens\': 0, \'reasoning_tokens\': 0, \'rejected_prediction_tokens\': 0}, \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cached_tokens\': 0}}, \'model_name\': \'gpt-4o-mini\', \'system_fingerprint\': \'fp_51db84afab\', \'finish_reason\': \'stop\', \'logprobs\': None} id=\'run--b5226a67-31a9-4253-bf61-630e21d8a23a-0\'', 'content=\'- **Ticker**: MMM\\n- **Company Name**: \\n- **Headcount Growth**: Decreased from **50,000** in 2023 to **36,000** in 2024.\\n- **Challenge Summary**: Headcount reduction impacting operational capacity.\\n- **Challenge Context**: Decrease in workforce affecting productivity.\\n- **Business Value**: \\n- **Business Value Context**: \\n- **Department**: \\n- **Adobe Solution**: \\n- **Filing Type**: 10-K\\n- **Filing Date**: 2025-02-05\\n- **Source Quote**: "Headcount decreased from 50,000 to 36,000."\\n- **Source Section**: Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations\' additional_kwargs={} response_metadata={\'token_usage\': {\'completion_tokens\': 167, \'prompt_tokens\': 2837, \'total_tokens\': 3004, \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \'audio_tokens\': 0, \'reasoning_tokens\': 0, \'rejected_prediction_tokens\': 0}, \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cached_tokens\': 0}}, \'model_name\': \'gpt-4o-mini\', \'system_fingerprint\': \'fp_51db84afab\', \'finish_reason\': \'stop\', \'logprobs\': None} id=\'run--7007df8a-b297-43cd-936c-b8e46e795bdb-0\'', 'content=\'- **Ticker**: MMM\\n- **Company Name**: 3M Company\\n- **Headcount Growth**: No relevant information\\n- **Challenge Summary**: Operational challenges may adversely affect financial performance.\\n- **Challenge Context**: Financial results depend on successful execution of business plans.\\n- **Business Value**: Improved operational efficiency and productivity.\\n- **Business Value Context**: Streamlining operations to enhance long-term performance.\\n- **Department**: No relevant information\\n- **Adobe Solution**: No relevant information\\n- **Filing Type**: 10-K\\n- **Filing Date**: 2025-02-05\\n- **Source Quote**: "The Company’s financial results depend on the successful execution of its business operating plans."\\n- **Source Section**: Item 1A. Risk Factors\' additional_kwargs={} response_metadata={\'token_usage\': {\'completion_tokens\': 168, \'prompt_tokens\': 2391, \'total_tokens\': 2559, \'completion_tokens_details\': {\'accepted_prediction_tokens\': 0, \'audio_tokens\': 0, \'reasoning_tokens\': 0, \'rejected_prediction_tokens\': 0}, \'prompt_tokens_details\': {\'audio_tokens\': 0, \'cached_tokens\': 1792}}, \'model_name\': \'gpt-4o-mini\', \'system_fingerprint\': \'fp_51db84afab\', \'finish_reason\': \'stop\', \'logprobs\': None} id=\'run--301057b4-a664-481d-a2e7-aca4e6c005f5-0\'']

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (3785624608.py, line 1)